# feat_crn_ensemble — Scoring-only ensemble (bureau + query -> final prediction)

Final step of the scoring batch pipeline:

```
COHORT -> feat_crn (bureau preds) -> feat_crn2 (query preds) -> feat_crn_ensemble -> final prediction
```

Reuses the **exact** `3_0_ensemble_model` / Jan-26 OOT methodology:
- inner join on `multibureau__c`
- `ensemble_prob = 0.7 * bureau_prob + 0.3 * query_prob`

**Scoring/inference only.** No labels, `Actual`, `Bucket`, PSI/KS, train/test/oot evaluation, or
model training. Does not modify `3_0_ensemble_model.ipynb` or `feat_crn3.ipynb`.

## 0a. Install pinned dependencies (run once)
Installs the exact runtime into **this** kernel. Comment the cell out after the first successful run. See `requirements.txt` for the pins and the rationale (xgboost 1.7.4, numpy<2.0).

In [ ]:
# Install the pinned runtime into THIS Jupyter kernel. The ensemble step only blends the two
# prediction CSVs, so it needs numpy<2.0 + pandas (no xgboost). Comment out after first run.
%pip install -q "numpy>=1.23,<2.0" "pandas>=2.1,<2.3"


## 0. Configuration — the only cell you edit

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================ EDIT ME ============================
COHORT = "mar_26"     # "<mon>_<yy>": must match the cohort used in feat_crn / feat_crn2
# =================================================================

# Ensemble weights — fixed to the applied 3_0_ensemble_model values (source of truth).
BUREAU_WEIGHT = 0.7
QUERY_WEIGHT  = 0.3

OUT_DIR = os.path.join(os.getcwd(), "output", COHORT)
BUREAU_PRED_CSV = os.path.join(OUT_DIR, f"bureau_predictions_{COHORT}.csv")   # from feat_crn.ipynb
QUERY_PRED_CSV  = os.path.join(OUT_DIR, f"query_predictions_{COHORT}.csv")    # from feat_crn2.ipynb
FINAL_OUT       = os.path.join(OUT_DIR, f"final_predictions_{COHORT}.csv")

# score column produced by each scoring notebook -> ensemble name (matches 3_0 rename of 'Prob')
BUREAU_SCORE_COL = "preclosure_score"          # feat_crn
QUERY_SCORE_COL  = "preclosure_score_query"    # feat_crn2
JOIN_KEY = "multibureau__c"

print("COHORT       :", COHORT)
print("bureau preds :", BUREAU_PRED_CSV)
print("query preds  :", QUERY_PRED_CSV)
print("weights      : bureau", BUREAU_WEIGHT, "| query", QUERY_WEIGHT)

## 1. Load the two prediction files (fail loudly if missing)

In [ ]:
for _p, _who in [(BUREAU_PRED_CSV, "feat_crn (bureau)"), (QUERY_PRED_CSV, "feat_crn2 (query)")]:
    if not os.path.exists(_p):
        raise FileNotFoundError(
            f"Missing prediction file for COHORT={COHORT}: {_p}\n"
            f"Run {_who} for COHORT={COHORT} first to produce it.")

df_bureau = pd.read_csv(BUREAU_PRED_CSV)
df_query  = pd.read_csv(QUERY_PRED_CSV)
print("bureau predictions :", df_bureau.shape)
print("query  predictions :", df_query.shape)

## 2. Validate columns, keys, and duplicates (fail loudly)

In [ ]:
def _validate(df, score_col, source_name):
    # join key present
    if JOIN_KEY not in df.columns:
        raise AssertionError(f"{source_name}: missing join key '{JOIN_KEY}'. columns={list(df.columns)}")
    # score column present
    if score_col not in df.columns:
        raise AssertionError(f"{source_name}: missing score column '{score_col}'. columns={list(df.columns)}")
    # non-empty
    if len(df) == 0:
        raise AssertionError(f"{source_name}: no rows.")
    # null keys
    n_null = df[JOIN_KEY].isna().sum()
    if n_null:
        raise AssertionError(f"{source_name}: {n_null} null '{JOIN_KEY}' value(s).")
    # duplicate keys would make the ensemble ambiguous (row explosion on merge)
    dups = df[JOIN_KEY].duplicated().sum()
    if dups:
        ex = df.loc[df[JOIN_KEY].duplicated(keep=False), JOIN_KEY].unique()[:5]
        raise AssertionError(
            f"{source_name}: {dups} duplicate '{JOIN_KEY}' value(s) -> ambiguous ensemble. "
            f"examples={list(ex)}. Each customer must appear once.")
    print(f"[validate] OK - {source_name}: {len(df):,} rows, unique '{JOIN_KEY}', has '{score_col}'")

_validate(df_bureau, BUREAU_SCORE_COL, "bureau predictions")
_validate(df_query,  QUERY_SCORE_COL,  "query predictions")

# normalize the join key and map score columns to the 3_0 names
df_bureau[JOIN_KEY] = df_bureau[JOIN_KEY].astype(str)
df_query[JOIN_KEY]  = df_query[JOIN_KEY].astype(str)
df_bureau = df_bureau.rename(columns={BUREAU_SCORE_COL: "bureau_prob"})
df_query  = df_query.rename(columns={QUERY_SCORE_COL:  "query_prob"})

## 3. Inner join on `multibureau__c` + weighted ensemble (exact 3_0 logic)
Inner join is intentional — it reproduces `3_0_ensemble_model` (`pd.merge(..., on='multibureau__c')`).
Customers present in only one model are dropped; no fallback is added.

In [ ]:
bureau_ids = set(df_bureau[JOIN_KEY])
query_ids  = set(df_query[JOIN_KEY])

df_final = pd.merge(df_bureau[[JOIN_KEY, "bureau_prob"]],
                    df_query[[JOIN_KEY, "query_prob"]],
                    on=JOIN_KEY, how="inner")

if len(df_final) == 0:
    raise AssertionError(
        f"Empty inner join for COHORT={COHORT}: no '{JOIN_KEY}' overlap between bureau "
        f"({len(bureau_ids):,}) and query ({len(query_ids):,}) predictions.")

# exact 3_0 combination
df_final["ensemble_prob"] = BUREAU_WEIGHT * df_final["bureau_prob"] + QUERY_WEIGHT * df_final["query_prob"]

# coverage report
n_bureau_only = len(bureau_ids - query_ids)
n_query_only  = len(query_ids - bureau_ids)
print(f"bureau predictions        : {len(df_bureau):,}")
print(f"query  predictions        : {len(df_query):,}")
print(f"final ensemble predictions: {len(df_final):,}")
print(f"bureau-only dropped (inner): {n_bureau_only:,}")
print(f"query-only  dropped (inner): {n_query_only:,}")
df_final.head()

## 4. Save final predictions

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
df_final = df_final[[JOIN_KEY, "bureau_prob", "query_prob", "ensemble_prob"]]
df_final.to_csv(FINAL_OUT, index=False)
print("saved ->", FINAL_OUT, df_final.shape)
print(df_final["ensemble_prob"].describe())

---
**Scoring only.** Inner join + `0.7*bureau + 0.3*query` reproduce `3_0_ensemble_model` exactly.
Output: `output/<cohort>/final_predictions_<cohort>.csv` (`multibureau__c, bureau_prob, query_prob, ensemble_prob`).
No `Actual`/labels, buckets, PSI/KS, train/test/oot, or model fitting. `3_0_ensemble_model.ipynb`
and `feat_crn3.ipynb` are untouched.